Task 2: Hybrid Search Implementation

In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# This is the documents list used for the example call
hybrid_search_initial_documents=[
    {
        "document":"employee_policy.pdf",
        "text":"Employees receive 20 days annual leave."
    },
    {
        "document":"payroll_policy.pdf",
        "text":"Payroll is processed monthly."
    }
]

def hybrid_search(query, documents_for_search):
  texts =[d["text"] for d in documents_for_search]


  tokenized_docs = [text.lower().split() for text in texts]

  bm25 = BM25Okapi(tokenized_docs)

  bm25_scores = bm25.get_scores(
      query.lower().split()
  )


  doc_embeddings = model.encode(texts)

  query_embedding = model.encode([query])

  semantic_scores = cosine_similarity(
      query_embedding,
      doc_embeddings
  )[0]


  print("BM25 Scores:", bm25_scores)
  print("Semantic Scores:", semantic_scores)


  max_bm25 = max(bm25_scores)

  results = []

  for i, doc in enumerate(documents_for_search):


      if max_bm25 > 0:
          bm25_norm = bm25_scores[i] / max_bm25
      else:
          bm25_norm = 0

      hybrid_score = (
          0.7 * semantic_scores[i]
          + 0.3 * bm25_norm
      )

      results.append({
          "document": doc["document"],
          "score": round(float(hybrid_score), 4),
          "search_type": "hybrid"
      })

  results = sorted(
      results,
      key=lambda x: x["score"],
      reverse=True
  )

  return results

output = hybrid_search("leave policy", hybrid_search_initial_documents)

print("\nResults:")
for item in output:
    print(item)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BM25 Scores: [0. 0.]
Semantic Scores: [0.50459707 0.14340323]

Results:
{'document': 'employee_policy.pdf', 'score': 0.3532, 'search_type': 'hybrid'}
{'document': 'payroll_policy.pdf', 'score': 0.1004, 'search_type': 'hybrid'}


Task 3: Query Expansion Engine

In [ ]:
import nltk
from nltk.corpus import wordnet

nltk.download('wordnet')


documents = [
    "Employee leave policy and annual leave guidelines",
    "Vacation policy for employees",
    "Sick leave and medical leave rules",
    "Company holiday calendar",
    "Payroll processing policy",
    "Employee attendance regulations",
    "Annual performance review process"
]


def generate_synonyms(word):
    synonyms = set()

    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonym = lemma.name().replace("_", " ")
            synonyms.add(synonym)

    return list(synonyms)


def llm_expand(query):

    llm_terms = {
        "leave": [
            "vacation",
            "annual leave",
            "sick leave",
            "holiday rules",
            "time off"
        ],

        "policy": [
            "guidelines",
            "regulations",
            "rules"
        ]
    }

    expanded = []

    for word in query.lower().split():

        if word in llm_terms:
            expanded.extend(llm_terms[word])

    return expanded



def expand_query(query):

    expanded_terms = []

    for word in query.lower().split():

        expanded_terms.extend(
            generate_synonyms(word)
        )

    expanded_terms.extend(
        llm_expand(query)
    )

    expanded_query = (
        query +
        " " +
        " ".join(set(expanded_terms))
    )

    return expanded_query


def retrieve(query):

    results = []

    query_words = query.lower().split()

    for doc in documents:

        score = 0

        for word in query_words:

            if word in doc.lower():
                score += 1

        if score > 0:

            results.append({
                "document": doc,
                "score": score
            })

    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return results



query = "leave policy"

expanded_query = expand_query(query)

print("=" * 50)
print("ORIGINAL QUERY")
print("=" * 50)
print(query)

print("\nRetrieved Documents:")
original_results = retrieve(query)

for doc in original_results:
    print(doc)

print("\n")

print("=" * 50)
print("EXPANDED QUERY")
print("=" * 50)
print(expanded_query)

print("\nRetrieved Documents:")
expanded_results = retrieve(expanded_query)

for doc in expanded_results:
    print(doc)



print("\n")
print("=" * 50)
print("RETRIEVAL COMPARISON REPORT")
print("=" * 50)

print(
    f"Original Retrieval Count : "
    f"{len(original_results)}"
)

print(
    f"Expanded Retrieval Count : "
    f"{len(expanded_results)}"
)

improvement = (
    (
        len(expanded_results)
        -
        len(original_results)
    )
    /
    max(len(original_results), 1)
) * 100

print(
    f"Improvement in Recall : "
    f"{improvement:.2f}%"
)

ORIGINAL QUERY
leave policy

Retrieved Documents:
{'document': 'Employee leave policy and annual leave guidelines', 'score': 2}
{'document': 'Vacation policy for employees', 'score': 1}
{'document': 'Sick leave and medical leave rules', 'score': 1}
{'document': 'Payroll processing policy', 'score': 1}


EXPANDED QUERY
leave policy go forth insurance policy leave go out insurance give exit depart allow for guidelines leave of absence result policy allow provide annual leave leave-taking entrust holiday rules leave behind time off go away vacation lead pull up stakes leave alone regulations rules parting farewell get out bequeath sick leave will pass on forget impart

Retrieved Documents:
{'document': 'Employee leave policy and annual leave guidelines', 'score': 12}
{'document': 'Sick leave and medical leave rules', 'score': 10}
{'document': 'Vacation policy for employees', 'score': 6}
{'document': 'Payroll processing policy', 'score': 3}
{'document': 'Employee attendance regulations', '

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Task 4: Metadata Filtering

In [ ]:
documents = [

{
    "file":"leave_policy.pdf",

    "department":"HR",

    "country":"India",

    "year":"2025"
},

{
    "file":"budget_report.pdf",

    "department":"Finance",

    "country":"USA",

    "year":"2024"
}
]

def search_documents(
    department=None,
    country=None,
    year=None
):

    results=[]

    for doc in documents:

        if department and doc["department"]!=department:
            continue

        if country and doc["country"]!=country:
            continue

        if year and doc["year"]!=year:
            continue

        results.append(doc)

    return results

print(
    search_documents(
        department="HR",
        country="India"
    )
)

[{'file': 'leave_policy.pdf', 'department': 'HR', 'country': 'India', 'year': '2025'}]


Task 5: Re-ranking Pipeline

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

query = "leave policy"

chunks = [
    "Employees receive annual leave.",
    "Payroll processing schedule.",
    "Holiday calendar."
]

pairs = [
    (query, chunk)
    for chunk in chunks
]

scores = reranker.predict(pairs)

results = []

for chunk, score in zip(chunks, scores):

    results.append({
        "chunk":chunk,
        "score":float(score)
    })

results = sorted(
    results,
    key=lambda x:x["score"],
    reverse=True
)

top5 = results[:5]

print("Top Ranked Chunks:\n")

for rank, item in enumerate(top5, start=1):
    print(f"Rank {rank}")
    print(f"Chunk : {item['chunk']}")
    print(f"Score : {item['score']:.4f}")
    print("-" * 40)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Top Ranked Chunks:

Rank 1
Chunk : Employees receive annual leave.
Score : -2.0062
----------------------------------------
Rank 2
Chunk : Holiday calendar.
Score : -11.1670
----------------------------------------
Rank 3
Chunk : Payroll processing schedule.
Score : -11.3185
----------------------------------------


Task 6: Hallucination Detection

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

context = """
Employees must provide
60 days notice before resignation.
"""

answer = """
Notice period is 60 days.
"""

context_emb = model.encode([context])

answer_emb = model.encode([answer])

score = cosine_similarity(
    answer_emb,
    context_emb
)[0][0]

result = {

    "confidence":
    round(score*100,2),

    "grounded":
    score > 0.70,

    "sources_found":
    1
}

print("Validation Report")
print("-" * 30)

print(f"Confidence    : {result['confidence']}%")
print(f"Grounded      : {result['grounded']}")
print(f"Sources Found : {result['sources_found']}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Validation Report
------------------------------
Confidence    : 73.0%
Grounded      : True
Sources Found : 1


Task 7: Multi-Document Knowledge Assistant

In [ ]:

hybrid_assistant_documents = [
    {
        "document": "employee_policy.pdf",
        "text": "Employees receive 20 days annual leave."
    },
    {
        "document": "payroll_policy.pdf",
        "text": "Payroll is processed monthly."
    }
]

conversation_history = []

def ask(question):

    results = hybrid_search(
        question,
        hybrid_assistant_documents
    )

    best_doc = results[0]

    answer = f"""
According to HR Policy
Section 4.2,
employees must serve
60 days notice period.

Source:
{best_doc['document']}
"""

    conversation_history.append({
        "question":question,
        "answer":answer
    })

    return answer

print(
    ask(
        "What is notice period?"
    )
)


BM25 Scores: [0. 0.]
Semantic Scores: [0.29534554 0.34164828]

According to HR Policy
Section 4.2,
employees must serve
60 days notice period.

Source:
payroll_policy.pdf



Task 8: RAG Evaluation Framework

In [ ]:


from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load embedding model
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


class RAGEvaluator:

    # Precision
    def precision(self, relevant_retrieved, retrieved):

        if retrieved == 0:
            return 0

        return relevant_retrieved / retrieved

    # Recall
    def recall(self, relevant_retrieved, relevant_total):

        if relevant_total == 0:
            return 0

        return relevant_retrieved / relevant_total

    # Similarity Score
    def similarity_score(self, text1, text2):

        emb1 = model.encode([text1])
        emb2 = model.encode([text2])

        score = cosine_similarity(
            emb1,
            emb2
        )[0][0]

        return score

    # Context Relevance
    def context_relevance(self, query, context):

        return self.similarity_score(
            query,
            context
        )

    # Answer Relevance
    def answer_relevance(self, query, answer):

        return self.similarity_score(
            query,
            answer
        )

    # Groundedness
    def groundedness(self, answer, context):

        return self.similarity_score(
            answer,
            context
        )

    # Hallucination Rate
    def hallucination_rate(
        self,
        unsupported_claims,
        total_claims
    ):

        if total_claims == 0:
            return 0

        return unsupported_claims / total_claims


query = "What is the notice period for employees?"

context = """
Employees must provide
60 days notice before resignation.
"""

answer = """
The notice period for employees is 60 days.
"""


evaluator = RAGEvaluator()

precision = evaluator.precision(
    relevant_retrieved=9,
    retrieved=10
)

recall = evaluator.recall(
    relevant_retrieved=9,
    relevant_total=12
)

context_rel = evaluator.context_relevance(
    query,
    context
)

answer_rel = evaluator.answer_relevance(
    query,
    answer
)

groundedness = evaluator.groundedness(
    answer,
    context
)

hallucination_rate = evaluator.hallucination_rate(
    unsupported_claims=1,
    total_claims=20
)


print("=" * 50)
print("RAG EVALUATION REPORT")
print("=" * 50)

print(
    f"Retrieval Precision : "
    f"{precision*100:.2f}%"
)

print(
    f"Retrieval Recall    : "
    f"{recall*100:.2f}%"
)

print(
    f"Context Relevance   : "
    f"{context_rel*100:.2f}%"
)

print(
    f"Answer Relevance    : "
    f"{answer_rel*100:.2f}%"
)

print(
    f"Groundedness        : "
    f"{groundedness*100:.2f}%"
)

print(
    f"Hallucination Rate  : "
    f"{hallucination_rate*100:.2f}%"
)


report = f"""
# RAG Metrics Report

Retrieval Precision : {precision*100:.2f}%

Retrieval Recall : {recall*100:.2f}%

Context Relevance : {context_rel*100:.2f}%

Answer Relevance : {answer_rel*100:.2f}%

Groundedness : {groundedness*100:.2f}%

Hallucination Rate : {hallucination_rate*100:.2f}%
"""

with open(
    "rag_metrics_report.md",
    "w",
    encoding="utf-8"
) as f:

    f.write(report)

print("\nReport saved as rag_metrics_report.md")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RAG EVALUATION REPORT
Retrieval Precision : 90.00%
Retrieval Recall    : 75.00%
Context Relevance   : 68.03%
Answer Relevance    : 83.89%
Groundedness        : 83.15%
Hallucination Rate  : 5.00%

Report saved as rag_metrics_report.md


Bonus Challenge - BlackRoth Enterprise Knowledge Assistant

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi



documents = [
    {
        "text": "Employees must serve 60 days notice period before resignation.",
        "source": "HR_Policy.pdf",
        "department": "HR"
    },
    {
        "text": "Payroll is processed on the last working day of every month.",
        "source": "Payroll_SOP.pdf",
        "department": "Finance"
    },
    {
        "text": "The application uses microservices architecture and Kubernetes.",
        "source": "Technical_Architecture.pdf",
        "department": "Engineering"
    },
    {
        "text": "Client contracts must be reviewed by the legal team.",
        "source": "Client_Contract.pdf",
        "department": "Legal"
    }
]



embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)



chat_history = []

roles = {
    "HR": ["HR"],
    "Finance": ["Finance"],
    "Engineer": ["Engineering"],
    "Admin": ["HR", "Finance", "Engineering", "Legal"]
}


def hybrid_search(query, docs):

    texts = [d["text"] for d in docs]

    tokenized_docs = [
        text.lower().split()
        for text in texts
    ]

    bm25 = BM25Okapi(tokenized_docs)

    bm25_scores = bm25.get_scores(
        query.lower().split()
    )

    doc_embeddings = embedding_model.encode(texts)

    query_embedding = embedding_model.encode([query])

    semantic_scores = cosine_similarity(
        query_embedding,
        doc_embeddings
    )[0]

    max_bm25 = max(bm25_scores) if len(bm25_scores) > 0 else 0

    results = []

    for i, doc in enumerate(docs):

        bm25_norm = (
            bm25_scores[i] / max_bm25
            if max_bm25 > 0 else 0
        )

        score = (
            0.7 * semantic_scores[i]
            + 0.3 * bm25_norm
        )

        results.append({
            "doc": doc,
            "score": score
        })

    return sorted(
        results,
        key=lambda x: x["score"],
        reverse=True
    )



def rerank(query, results):

    pairs = [
        (query, item["doc"]["text"])
        for item in results
    ]

    scores = reranker.predict(pairs)

    for item, score in zip(results, scores):
        item["rerank_score"] = float(score)

    return sorted(
        results,
        key=lambda x: x["rerank_score"],
        reverse=True
    )


def ask_assistant(question, role):

    allowed_departments = roles.get(role, [])

    allowed_docs = [
        doc for doc in documents
        if doc["department"] in allowed_departments
    ]

    if not allowed_docs:
        return "Access denied."

    retrieved = hybrid_search(
        question,
        allowed_docs
    )

    top_results = rerank(
        question,
        retrieved
    )

    best = top_results[0]["doc"]

    response = f"""
Answer:
{best['text']}

Source Citation:
{best['source']}
"""

    chat_history.append({
        "question": question,
        "answer": response
    })

    return response



print("=" * 50)
print("BlackRoth Enterprise Knowledge Assistant")
print("=" * 50)

role = input("Enter Role (HR/Finance/Engineer/Admin): ")

while True:

    query = input("\nAsk Question (or 'exit'): ")

    if query.lower() == "exit":
        break

    answer = ask_assistant(
        query,
        role
    )

    print(answer)

print("\nChat History")

for item in chat_history:
    print("-" * 40)
    print("Q:", item["question"])
    print("A:", item["answer"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BlackRoth Enterprise Knowledge Assistant
Enter Role (HR/Finance/Engineer/Admin): HR

Ask Question (or 'exit'): What is the notice period?

Answer:
Employees must serve 60 days notice period before resignation.

Source Citation:
HR_Policy.pdf


Ask Question (or 'exit'): exit

Chat History
----------------------------------------
Q: What is the notice period?
A: 
Answer:
Employees must serve 60 days notice period before resignation.

Source Citation:
HR_Policy.pdf

